# Training Evaluation Results

In [16]:
import csv
import os

import numpy as np


RESULTS_DIR = 'results'
EXPERIMENTS = 'model', 'online'
EXTINCTIONS = 80, 200, 1000
CAMERAS = 'front', 'turntable'

In [17]:
col = {'model': 'time_ms', 'online': 'train_time_ms'}
train_times = {}
volumes = {}

for experiment in EXPERIMENTS:
    train_times[experiment] = {}
    volumes[experiment] = set()
    for camera in CAMERAS:
        train_times[experiment][camera] = {}
        for filename in os.listdir(os.path.join(RESULTS_DIR, experiment, camera)):
            if not filename.endswith('.csv'):
                continue
            volume, extinction, transfer_function = filename[:-4].rsplit('_', 2)
            volumes[experiment].add(volume)
            with open(
                os.path.join(RESULTS_DIR, experiment, camera, filename), 'r'
            ) as f:
                reader = csv.DictReader(f)
                data = [{k: float(v) for k, v in row.items()} for row in reader]
                times = [d[col[experiment]] for d in data]
            train_times[experiment][camera][
                volume, int(extinction), int(transfer_function)
            ] = times

volumes = sorted(set.intersection(*volumes.values()))

## Train time per transfer function

In [21]:
print(
    f'{"Volume":<15} {"Extinction":<12} {"TF":<4} {"Model Time [ms]":<20} {"Online Time [ms]":<20}'
)
for volume in volumes:
    for extinction in EXTINCTIONS:
        for tf in 1, 2, 3:
            model_mean = np.mean(train_times['model']['front'][volume, extinction, tf])
            model_se = np.std(
                train_times['model']['front'][volume, extinction, tf]
            ) / np.sqrt(len(train_times['model']['front'][volume, extinction, tf]))
            online_mean = np.mean(
                train_times['online']['front'][volume, extinction, tf]
            )
            online_se = np.std(
                train_times['online']['front'][volume, extinction, tf]
            ) / np.sqrt(len(train_times['online']['front'][volume, extinction, tf]))

            model_str = f'{model_mean:6.2f} ± {model_se:5.2f}'
            online_str = f'{online_mean:6.2f} ± {online_se:5.2f}'
            print(
                f'{volume:<15} {extinction:<12} {tf:<4} {model_str:<20} {online_str:<20}'
            )

Volume          Extinction   TF   Model Time [ms]      Online Time [ms]    
chameleon       80           1     10.56 ±  0.17        21.92 ±  0.03      
chameleon       80           2      8.58 ±  0.03        17.38 ±  0.03      
chameleon       80           3     10.84 ±  0.02        23.18 ±  0.02      
chameleon       200          1     11.88 ±  0.02        30.31 ±  0.03      
chameleon       200          2     11.54 ±  0.02        34.21 ±  0.04      
chameleon       200          3     12.99 ±  0.02        38.30 ±  0.01      
chameleon       1000         1     12.71 ±  0.02        37.63 ±  0.03      
chameleon       1000         2     13.96 ±  0.02        44.16 ±  0.04      
chameleon       1000         3     13.85 ±  0.03        41.97 ±  0.04      
mri_ventricles  80           1     12.34 ±  0.11        20.24 ±  0.01      
mri_ventricles  80           2     14.96 ±  0.01        24.69 ±  0.01      
mri_ventricles  80           3     16.56 ±  0.02        26.72 ±  0.01      
mri_ventricl

## Train time per extinction only

In [22]:
print(
    f'{"Volume":<15} {"Extinction":<12} {"Model Time [ms]":<20} {"Online Time [ms]":<20}'
)
for volume in volumes:
    for extinction in EXTINCTIONS:
        model_times = np.concatenate(
            [train_times['model']['front'][volume, extinction, tf] for tf in (1, 2, 3)]
        )
        online_times = np.concatenate(
            [train_times['online']['front'][volume, extinction, tf] for tf in (1, 2, 3)]
        )

        model_mean = np.mean(model_times)
        model_se = np.std(model_times) / np.sqrt(len(model_times))
        online_mean = np.mean(online_times)
        online_se = np.std(online_times) / np.sqrt(len(online_times))

        model_str = f'{model_mean:6.2f} ± {model_se:5.2f}'
        online_str = f'{online_mean:6.2f} ± {online_se:5.2f}'
        print(f'{volume:<15} {extinction:<12} {model_str:<20} {online_str:<20}')

Volume          Extinction   Model Time [ms]      Online Time [ms]    
chameleon       80             9.99 ±  0.06        20.62 ±  0.04      
chameleon       200           12.13 ±  0.02        34.02 ±  0.06      
chameleon       1000          13.51 ±  0.02        41.01 ±  0.05      
mri_ventricles  80            14.62 ±  0.06        23.32 ±  0.04      
mri_ventricles  200           17.78 ±  0.02        30.15 ±  0.02      
mri_ventricles  1000          19.82 ±  0.01        41.85 ±  0.02      
